In [1]:
import pandas as pd
import numpy as np
import os
import swifter
from scipy.stats import wasserstein_distance
import ast
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', None)

C:\Users\isen1\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Get the distributions across 100 runs and see if they match the logprobs (WD)

In [2]:
versions = True

In [3]:
import opinionqa_helpers as ph
from opinionqa_helpers_custom import compute_match, compute_alignment, get_probs

In [4]:
LLMs = [# "merged_results_user_OLMo-2-0325-32B-Instruct",
        "merged_results_user_OLMo-2-1124-7B-Instruct",
        "merged_results_user_Llama-3.1-8B-Instruct",
        # "merged_results_user_Llama-3.3-70B-Instruct",
        # "merged_results_user_gemma-3-27b-it"
       ]

In [5]:
opinionqa_human_response_path = "../../opinions_qa/data/human_resp/"
runs_path = "../data/QA_100_runs/"
llm_result_path = "../data/results_qa/"

ATP = "American_Trends_Panel_"
genders = ['Male', 'Female']
races = ['White', 'Asian', 'Hispanic', 'Black'] 

RESULT_DIR = '../../opinions_qa/data/runs'

In [6]:
data_runs = {}
for llm in LLMs:
    data_runs[llm] = pd.read_csv(runs_path + llm + '.csv')

In [7]:
data_runs[llm][['qa_key', 'answer_option', 'qa_options']]

,qa_key,answer_option,qa_options
0,INEQ8_g_W54,A,"('A', 'B', 'C', 'D', 'E')"
1,INEQ8_h_W54,D,"('A', 'B', 'C', 'D', 'E')"
2,INEQ8_i_W54,A,"('A', 'B', 'C', 'D', 'E')"
3,INEQ8_j_W54,A,"('A', 'B', 'C', 'D', 'E')"
4,INEQ11_W54,A,"('A', 'B', 'C')"
...,...,...,...
1435425,LEGALIMMIGAMT_W92,E,"('A', 'B', 'C', 'D', 'E', 'F')"
1435426,UNIMMIGCOMM_W92,A,"('A', 'B', 'C', 'D', 'E')"
1435427,GODMORALIMP_W92,D,"('A', 'B', 'C', 'D', 'E')"
1435428,REPRSNTREP_W92,D,"('A', 'B', 'C', 'D', 'E')"


In [8]:
data_log = {}

for llm in LLMs:
    data_log[llm] = pd.read_csv(llm_result_path+llm+'.csv')

data_log[llm]['qa_key'][:5]

0    INEQ8_g_W54
1    INEQ8_h_W54
2    INEQ8_i_W54
3    INEQ8_j_W54
4     INEQ11_W54
Name: qa_key, dtype: object

### get logprobs

In [9]:
for llm in LLMs:
    D_Ms, R_Ms = [], []
    df = data_log[llm].copy()
    for n, row in df.iterrows():
        lps = ast.literal_eval(df['answer_dist_clean'][n])
        options = ast.literal_eval(df['qa_options'][n])
        res = ph.get_probabilities_custom(lps, options)
        ordinal = len(options) - 1
        D_Ms.append(res['probs_unnorm'][:ordinal] / np.sum(res['probs_unnorm'][:ordinal]))
        R_Ms.append(np.sum(res['probs_norm'][ordinal:]))
    data_log[llm]['D_M'] = D_Ms
    data_log[llm]['R_M'] = R_Ms

In [10]:
qa_demo_map = {'F': 'Female',
               'M' : 'Male'}

genders = ['F', 'M']
persona_types = data_log[llm]['persona_type'].unique()
persona_strings = data_log[llm]['persona_string'].unique()
prompt_versions = data_log[llm]['prompt_version'].unique()
qa_keys = data_log[llm]['qa_key'].unique()

### get ditributions from 100 runs

In [11]:
def normalize(data):
    total = sum(data.values())
    normalized = {k: v / total for k, v in data.items()}
    return normalized

In [12]:
runs_df_rows = []

for llm in LLMs:
    print(llm)
    df = data_runs[llm].copy()
    for gender in genders:
        model_df_subset = df[df['gender'] == gender]
        
        
        for race in races:
            model_df_subset_race = model_df_subset[model_df_subset['race'] == race]
            
            for persona_type in ['dem_cat+descr', 'dem_descr', 'name']:
                model_df_subset_pt = model_df_subset_race[model_df_subset_race['persona_type']\
                                                   == persona_type]
                
                for persona_string in persona_strings:
                    model_df_subset_ps = model_df_subset_pt[model_df_subset_pt['persona_string']\
                                                        == persona_string]
                    
                    for prompt_version in prompt_versions:
                            model_df_subset_pv = model_df_subset_ps[model_df_subset_ps['prompt_version'] == \
                                                      prompt_version]
                            
                            for qa_key in qa_keys:
                                model_df_subset_qk = model_df_subset_pv[model_df_subset_pv['qa_key'] == qa_key]
                                
                                try:
                                    options = sorted(list(ast.literal_eval(model_df_subset_qk['qa_options'].values[0])))
                                    dist_key = model_df_subset_qk.groupby('answer_option').size().to_dict()
                                
                                    # remove the refusal (the last option)
                                    final_dist_key = {}
                                
                                    for k in options[:-1]:
                                        if k in dist_key:
                                            final_dist_key[k] = dist_key[k]
                                        else:
                                            final_dist_key[k] = 0
                                    fdk = normalize(final_dist_key.copy()).values()
                                
                                    if options[-1] in dist_key:
                                        refusal = dist_key[options[-1]] / 100
                                    else:
                                        refusal = 0
                                
                                    runs_df_rows.append([llm, gender, race, persona_type,
                                        persona_string, prompt_version, qa_key,
                                        fdk, refusal])
                                except:
                                    pass

merged_results_user_OLMo-2-1124-7B-Instruct
merged_results_user_Llama-3.1-8B-Instruct


In [13]:
runs_df_rows[0]

['merged_results_user_OLMo-2-1124-7B-Instruct',
 'F',
 'White',
 'dem_cat+descr',
 '2nd',
 'v1',
 'INEQ8_g_W54',
 dict_values([0.6161616161616161, 0.3434343434343434, 0.04040404040404041, 0.0]),
 0.01]

In [14]:
temp = pd.DataFrame(runs_df_rows)
temp.columns = ['llm', 'gender', 'race', 'persona_type',
                                        'persona_string', 'prompt_version', 'qa_key',
                                        'D_H', 'R_H']
runs_df_dict = {}
for llm in LLMs:
    runs_df_dict[llm] = temp[temp['llm'] == llm]

In [15]:
runs_df_dict['merged_results_user_Llama-3.1-8B-Instruct']

,llm,gender,race,persona_type,persona_string,prompt_version,qa_key,D_H,R_H
14400,merged_results_user_Llama-3.1-8B-Instruct,F,White,dem_cat+descr,2nd,v1,INEQ8_g_W54,"(1.0, 0.0, 0.0, 0.0)",0.0
14401,merged_results_user_Llama-3.1-8B-Instruct,F,White,dem_cat+descr,2nd,v1,INEQ8_h_W54,"(0.0, 0.0, 0.0, 1.0)",0.0
14402,merged_results_user_Llama-3.1-8B-Instruct,F,White,dem_cat+descr,2nd,v1,INEQ8_i_W54,"(1.0, 0.0, 0.0, 0.0)",0.0
14403,merged_results_user_Llama-3.1-8B-Instruct,F,White,dem_cat+descr,2nd,v1,INEQ8_j_W54,"(1.0, 0.0, 0.0, 0.0)",0.0
14404,merged_results_user_Llama-3.1-8B-Instruct,F,White,dem_cat+descr,2nd,v1,INEQ11_W54,"(0.53, 0.47)",0.0
...,...,...,...,...,...,...,...,...,...
28756,merged_results_user_Llama-3.1-8B-Instruct,M,Black,name,interview,v2,LEGALIMMIGAMT_W92,"(0.0, 0.04, 0.96, 0.0, 0.0)",0.0
28757,merged_results_user_Llama-3.1-8B-Instruct,M,Black,name,interview,v2,UNIMMIGCOMM_W92,"(0.86, 0.14, 0.0, 0.0)",0.0
28758,merged_results_user_Llama-3.1-8B-Instruct,M,Black,name,interview,v2,GODMORALIMP_W92,"(0.0, 0.5161290322580645, 0.3225806451612903, 0.16129032258064516)",0.0
28759,merged_results_user_Llama-3.1-8B-Instruct,M,Black,name,interview,v2,REPRSNTREP_W92,"(0.0, 0.0, 0.13, 0.87)",0.0


In [16]:
data_runs['merged_results_user_Llama-3.1-8B-Instruct'].groupby('qa_key').size().sort_values()

qa_key
FREECOLL_W92      12530
GAP21Q34_e_W82    13280
GAP21Q34_b_W82    13630
PROG_RNEED_W92    13830
GAP21Q34_a_W82    14180
                  ...  
GAP21Q19_d_W82    14400
GAP21Q19_c_W82    14400
GAP21Q19_b_W82    14400
GAP21Q17_W82      14400
WORRY2e_W54       14400
Length: 100, dtype: int64

### find alignment

In [17]:
def compute_alignment__(human_df_subset, model_df_subset = False, baseline =  False):
    alignment = []
    if not baseline:
        all_qkeys = set(model_df_subset['qa_key'].unique())
        all_qkeys = list(all_qkeys.intersection(human_df_subset['qa_key'].unique()))
        for qkey in all_qkeys:
            d1 = model_df_subset[model_df_subset['qa_key'] == qkey]['D_M'].values[0]
            d2 = list(human_df_subset[human_df_subset['qa_key'] == qkey]['D_H'].values[0])
            # print(d1)
            # print(d2)
            wd_m = 1 - (wasserstein_distance(d1, d2) / (len(d1)-1)) # this metric is a bit sketchy
            wd_m = wasserstein_distance(d1, d2) # used in https://arxiv.org/pdf/2502.16761
            alignment.append(wd_m)
    else:
        ## upper bound for wd
        all_qkeys = human_df_subset['qa_key'].unique()
        for qkey in all_qkeys:
            d2 = human_df_subset[human_df_subset['qa_key'] == qkey]['D_H'].values[0]
            d1 = np.random.dirichlet(np.ones(len(d2)-1)).tolist() # random
            wd_m = 1 - (wasserstein_distance(d1, d2) / (len(d1)-1)) # this metric is a bit sketchy
            wd_m = wasserstein_distance(d1, d2) # used in https://arxiv.org/pdf/2502.16761
            alignment.append(wd_m)
        
    
    return(np.mean(alignment))

In [18]:
dist_align_dfs = {}
for llm in LLMs:
    print(llm)
    alignment_df_rows = []
    df = data_log[llm]
    run_df = runs_df_dict[llm]
    for gender in genders:
        model_df_subset = df[df['gender'] == gender]
        run_df_subset = run_df[run_df['gender'] == gender] # replace with the correponding 100 run 
    
        for race in races:
            model_df_subset_race = model_df_subset[model_df_subset['race'] == race]
            run_df_subset_r = run_df_subset[run_df_subset['race'] == race]
        
            for persona_type in ['dem_cat+descr', 'dem_descr', 'name']:
                model_df_subset_pt = model_df_subset_race[model_df_subset_race['persona_type']\
                                                   == persona_type]
                run_df_subset_pt = run_df_subset_r[run_df_subset_r['persona_type']\
                                                   == persona_type]
            
                
            
                for persona_string in persona_strings:
                    model_df_subset_ps = model_df_subset_pt[model_df_subset_pt['persona_string']\
                                                        == persona_string]
                    run_df_subset_ps = run_df_subset_pt[run_df_subset_pt['persona_string']\
                                                        == persona_string]
                
                    if versions:
                        for prompt_version in prompt_versions:
                            model_df_subset_pv = model_df_subset_ps[model_df_subset_ps['prompt_version'] == \
                                                      prompt_version]
                            run_df_subset_pv = run_df_subset_ps[run_df_subset_ps['prompt_version'] == \
                                                      prompt_version]
                            
                            alignment = compute_alignment__(run_df_subset_pv,
                                                      model_df_subset_pv)
                            alignment_df_row = [gender, race, persona_type,
                                        persona_string, prompt_version,
                                        alignment]
                            alignment_df_rows.append(alignment_df_row)
                    else:
                        alignment = compute_alignment__(runs_df_subset_ps,
                                                      model_df_subset_ps)
                        alignment_df_row = [gender, race, persona_type,
                                        persona_string, # prompt_version,
                                        alignment]
                        alignment_df_rows.append(alignment_df_row)
                        
                    

    alignment_df = pd.DataFrame(alignment_df_rows)
    if versions:
        alignment_df.columns = ['gender', 'race', 'persona_type',
                        'persona_string', 'prompt_version',
                        'alignment']
    else:
        alignment_df.columns = ['gender', 'race', 'persona_type',
                        'persona_string', # 'prompt_version',
                        'alignment']
    dist_align_dfs[llm] =  alignment_df

merged_results_user_OLMo-2-1124-7B-Instruct
merged_results_user_Llama-3.1-8B-Instruct


In [26]:
dist_align_dfs['merged_results_user_OLMo-2-1124-7B-Instruct'].mean(numeric_only=True)

alignment    0.034849
dtype: float64

In [27]:
dist_align_dfs['merged_results_user_OLMo-2-1124-7B-Instruct'].std(numeric_only=True)

alignment    0.024999
dtype: float64

In [28]:
dist_align_dfs['merged_results_user_Llama-3.1-8B-Instruct'].mean(numeric_only=True)

alignment    0.062646
dtype: float64

In [29]:
dist_align_dfs['merged_results_user_Llama-3.1-8B-Instruct'].std(numeric_only=True)

alignment    0.005876
dtype: float64